# RingVoz — Modelado predictivo de fraude

**Integrantes:** José Becerra · Marleny Guiral Marín · Geraldine Suárez  
**Responsable de este notebook:** José Becerra

Modelo predictivo supervisado para la variable `is_fraud`. Se entrenan y comparan **tres métodos** (Random Forest, XGBoost, MLP) sobre el mismo split y preprocesamiento. Criterio de selección: **PR-AUC** en el conjunto de test (apropiado para clase desbalanceada).

- Entrada: `dataset_fraude_ringvoz_class_last.arff` (8 predictoras + `is_fraud`).
- Salida: modelo serializado + JSON de métricas + reportes por método, en Drive.
- Variables predictoras (todas conocidas antes de la respuesta del gateway, sin leakage): `Account_Type`, `Amount`, `Transaction_Type`, `Merchant`, `Transaction_Use`, `pmntMethodId`, `hour_of_day`, `day_of_week`.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from scipy.io import arff

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, classification_report,
                             f1_score, recall_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

BASE_PATH = '/content/'
ARFF_PATH = BASE_PATH + 'dataset_fraude_ringvoz_class_last.arff'
TARGET = 'is_fraud'
RANDOM_STATE = 42

In [ ]:
data, _ = arff.loadarff(ARFF_PATH)
df = pd.DataFrame(data)
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].str.decode('utf-8')

y = (df[TARGET].astype(str) == '1').astype(int)
X = df.drop(columns=[TARGET])
print(f'Filas: {len(df):,} | Predictoras: {list(X.columns)}')
print(f'Positivos clase 1: {int(y.sum())} ({y.mean()*100:.3f} %) — desbalance extremo')

## Split estratificado y preprocesamiento

Split 75/25 estratificado por `is_fraud` para que los positivos se repartan proporcionalmente entre train y test. `scale_pos_weight` de XGBoost y `class_weight='balanced_subsample'` de RF se calibran al desbalance del train.

In [ ]:
cat_cols = ['Account_Type','Transaction_Type','Merchant','Transaction_Use','day_of_week']
num_cols = ['Amount','pmntMethodId','hour_of_day']

def build_pre():
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', StandardScaler(), num_cols),
    ])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

neg, pos = int((y_train==0).sum()), int((y_train==1).sum())
spw = neg / max(pos, 1)
print(f'Train: {len(y_train):,} (pos={pos}) | Test: {len(y_test):,} (pos={int(y_test.sum())}) | scale_pos_weight={spw:.1f}')

In [ ]:
models = {
    'RandomForest': Pipeline([
        ('prep', build_pre()),
        ('clf', RandomForestClassifier(n_estimators=180, max_depth=18, min_samples_leaf=2,
                                       class_weight='balanced_subsample',
                                       random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    'XGBoost': Pipeline([
        ('prep', build_pre()),
        ('clf', XGBClassifier(n_estimators=600, max_depth=5, min_child_weight=2,
                              learning_rate=0.04, subsample=0.85, colsample_bytree=0.85,
                              reg_lambda=1.2, gamma=0.05, scale_pos_weight=spw,
                              random_state=RANDOM_STATE, eval_metric='logloss',
                              tree_method='hist', n_jobs=-1)),
    ]),
    'MLP': Pipeline([
        ('prep', build_pre()),
        ('clf', MLPClassifier(hidden_layer_sizes=(96,48), activation='relu', max_iter=300,
                              alpha=5e-4, learning_rate_init=0.001, batch_size=2048,
                              random_state=RANDOM_STATE, early_stopping=True,
                              validation_fraction=0.12, n_iter_no_change=20)),
    ]),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    results[name] = {
        'roc_auc': roc_auc_score(y_test, proba),
        'pr_auc': average_precision_score(y_test, proba),
        'recall_clase_1': recall_score(y_test, pred),
        'f1_macro': f1_score(y_test, pred, average='macro'),
        'pred': pred, 'proba': proba,
    }
    print(f'{name}: PR-AUC={results[name]["pr_auc"]:.4f}  ROC-AUC={results[name]["roc_auc"]:.4f}  '
          f'Recall1={results[name]["recall_clase_1"]:.3f}  F1m={results[name]["f1_macro"]:.4f}')

## Comparación de métodos en test

In [ ]:
tabla = pd.DataFrame({k: {m: v for m, v in r.items() if m in ('pr_auc','roc_auc','recall_clase_1','f1_macro')}
                      for k, r in results.items()}).T.round(4)
tabla

**Lectura de la tabla.**

- **Random Forest** obtiene la mayor **PR-AUC (0,0122)** y la mayor ROC-AUC (0,9079). Es el modelo elegido.
- **XGBoost** queda muy cerca en PR-AUC (0,0120) con ROC-AUC 0,8449. Diferencia de 0,0002 — virtualmente empate en la métrica decisiva.
- **MLP** colapsa: PR-AUC 0,0002 (orden de la tasa base) y recall de la clase 1 = 0,0. Con solo 17 positivos no consigue señal de gradiente útil; los árboles manejan mejor el desbalance extremo.
- **PR-AUC en perspectiva**: la tasa base de positivos es 0,000189 (17 / 89.890). Un PR-AUC de 0,0122 es ≈ **65× la tasa base**, lo que indica que el modelo ordena los positivos sobre los negativos muy por encima del azar aunque el valor absoluto suene bajo. Esa comparación contra la tasa base es la lectura correcta en problemas con clase positiva ínfima.

In [ ]:
best_name = max(results, key=lambda k: results[k]['pr_auc'])
print('Mejor modelo (por PR-AUC):', best_name)
print('\nMatriz de confusión en test —', best_name)
cm = confusion_matrix(y_test, results[best_name]['pred'])
print(pd.DataFrame(cm, index=['real_0','real_1'], columns=['pred_0','pred_1']))
print('\nClassification report —', best_name)
print(classification_report(y_test, results[best_name]['pred'], digits=3,
                            target_names=['clase_0','clase_1'], zero_division=0))

**Interpretación de la matriz de confusión (Random Forest).** Con umbral por defecto (0,5), el modelo recupera **el 25 % de los fraudes del test** (1 de cada 4 positivos del conjunto de test). El recall se puede subir bajando el umbral, a costa de aumentar falsos positivos — la decisión final depende del costo relativo en producción entre un fraude no detectado y una transacción legítima rechazada. Para la evaluación académica se reporta el umbral 0,5 sin calibrar.

## Persistencia de artefactos en Drive

In [ ]:
best_pipe = models[best_name]

joblib.dump(best_pipe, BASE_PATH + 'modelo_riesgo.joblib')

domains = {c: sorted(X[c].astype(str).unique().tolist()) for c in cat_cols}
domains['_amount_min'] = float(X['Amount'].min())
domains['_amount_max'] = float(X['Amount'].max())
with open(BASE_PATH + 'feature_domains.json', 'w', encoding='utf-8') as f:
    json.dump(domains, f, ensure_ascii=False, indent=2)

meta = {
    'mejor_modelo': best_name,
    'motivo': 'Mayor PR-AUC en conjunto de test (clase minoritaria).',
    'metricas_test': {k: {m: float(v) for m, v in r.items()
                          if m in ('roc_auc','pr_auc','recall_clase_1','f1_macro')}
                      for k, r in results.items()},
    'n_registros': int(len(df)),
    'positivos_clase_1': int(y.sum()),
    'variable_objetivo': 'is_fraud (clase 1 = positivos en el ARFF de entrenamiento).',
}
with open(BASE_PATH + 'resultados_modelado.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

for name, r in results.items():
    rep = classification_report(y_test, r['pred'], digits=3,
                                target_names=['clase_0','clase_1'], zero_division=0)
    with open(BASE_PATH + f'report_{name}.txt', 'w', encoding='utf-8') as f:
        f.write(rep)

print('Artefactos guardados en', BASE_PATH)

## Conclusiones

- **Mejor modelo: Random Forest** (PR-AUC test = 0,0122, ROC-AUC = 0,9079, recall clase 1 = 0,250 con umbral 0,5).
- **XGBoost** queda en empate técnico con RF (PR-AUC 0,0120). Es el suplente natural si en producción se prefiere su mayor capacidad de calibración de probabilidades.
- **MLP** no es adecuado para este problema: con 17 positivos no genera señal de aprendizaje útil. Se reporta para cumplir la rúbrica de tres métodos comparados, no como candidato.
- **PR-AUC ≈ 65× la tasa base** confirma que el modelo aprende señal real; el valor absoluto bajo refleja la dificultad estructural del problema (clase 0,02 %).
- **Limitaciones reportables en el informe:** (1) etiqueta proxy basada en reglas del gateway, no en confirmación jurídica de fraude; (2) umbral 0,5 no calibrado por costo de error; (3) split aleatorio, no temporal — en producción conviene validación por fecha para detectar concept drift.
- **Artefactos generados en Drive:** `modelo_riesgo.joblib`, `feature_domains.json`, `resultados_modelado.json`, `report_RandomForest.txt`, `report_XGBoost.txt`, `report_MLP.txt`. Son los inputs de la app Streamlit del despliegue.
